# 002 · Transformer → GRU: владение по трекингу
Запуск: выберите Python с установленными зависимостями из `requirements.txt`, затем **Run All**.
Прогресс показывает `tqdm`: подготовка данных, эпохи, блоки кадров, прогноз матча.

По умолчанию: 12 эпох на первых 4 минутах, диагностика на последней минуте;
затем новая модель с нуля, 12 эпох на всех 5 минутах и прогноз полного матча.
Это небольшая экспериментальная сеть, не обещание улучшения baseline. 17 владений недостаточно для надёжной оценки обобщения.
Проверочная минута уже просматривалась в предыдущих экспериментах: это диагностический, а не независимый тест.

Все результаты, веса и журналы сохраняются **локально**, вне Git. Входные координаты не печатаются.
`history.csv` обновляется после каждой эпохи, `progress.json` — после каждого блока.
Повторный запуск продолжает с последней **завершённой эпохи**. Для нового запуска измените `RUN_NAME`.
Точное воспроизведение ожидается в одинаковом окружении на CPU; версии и конфигурация записываются.


In [ ]:
from pathlib import Path
import os, sys, json, time, random, hashlib, platform
import numpy as np
import pandas as pd
import torch
from torch import nn
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "games.py").exists())
sys.path.insert(0, str(ROOT))
from games import GAME, GAMES, asset_paths, half_video_blocks
from starter.baseline import team_of_each_track
from evaluation import possession_metrics as metrics

# CPU is the reproducible default; CUDA can be selected explicitly.
# CPU — воспроизводимый вариант по умолчанию; CUDA можно включить вручную.
DEVICE = "cpu"
RUN_NAME = "run_001"
EPOCHS = 12
SEED = 42
SEQ_LEN = 125                 # 5 seconds / 5 секунд на шаг градиента
SMOOTH_WEIGHT = 0.05
LEARNING_RATE = 3e-4
RUN_FINAL = True              # Refit on all 5 min / Обучить заново на всех 5 минутах
RUN_FULL_MATCH = True         # Predict after final training / Прогноз после финального обучения
RESUME = True
SMOKE = os.environ.get("POSSESSION_SMOKE") == "1"  # Developer check / Короткая проверка кода
if SMOKE:
    RUN_NAME, EPOCHS, SEQ_LEN = "smoke_test", 1, 32
    RUN_FINAL, RUN_FULL_MATCH = False, False
OUT = ROOT / "local_review/experiments/002_transformer_gru" / RUN_NAME
OUT.mkdir(parents=True, exist_ok=True)
CACHE = ROOT / "local_review/experiments/002_transformer_gru/features_v1"
CACHE.mkdir(parents=True, exist_ok=True)
PATHS = asset_paths()
N, FPS = GAMES[GAME]["n_video_frames"], GAMES[GAME]["fps"]
LENGTH, WIDTH = GAMES[GAME]["pitch"]
BLOCKS = half_video_blocks(GAME)
LABELS = np.array(["H", "A", "D"])
torch.set_num_threads(min(4, os.cpu_count() or 1))
torch.use_deterministic_algorithms(True)
def seed_all():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
seed_all()

def save_json(path, value):
    # Atomic replacement prevents half-written status files.
    # Атомарная замена защищает журнал от частичной записи.
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False))
    tmp.replace(path)

def digest(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

config = dict(seed=SEED, epochs=EPOCHS, seq_len=SEQ_LEN, smooth_weight=SMOOTH_WEIGHT,
              learning_rate=LEARNING_RATE, device=DEVICE, d_model=64, heads=4, layers=2,
              gru_hidden=64, dropout=0.1, features_version=1, smoke=SMOKE,
              torch=torch.__version__, numpy=np.__version__, python=platform.python_version())
config_path = OUT / "config.json"
if config_path.exists():
    assert json.loads(config_path.read_text()) == config, "Config changed: use a new RUN_NAME"
save_json(config_path, config)
print("Device:", DEVICE, "| Results:", OUT)


## Данные и признаки
Читаем исходный ATD; `frame_id - 1` совпадает с индексом видео. Флаги из предыдущего эксперимента воспроизводим здесь автоматически — отдельный локальный CSV не требуется.
Самая глубокая точка считается вратарём только на глубине 0–16,5 м от своих ворот; ограничение продольное. Подозрительные точки не удаляем.

11 признаков токена: x, y, dx/dy относительно мяча, наличие мяча, команда H/A, тип «мяч», предполагаемый вратарь, подозрительная точка, наличие наблюдения.
Координаты масштабируем фиксированными размерами поля; статистики проверки не используются.
Номера слотов и время матча не являются признаками. Отсутствующие игроки маскируются, а токен отсутствующего мяча остаётся с явным признаком пропуска.


In [ ]:
teams = team_of_each_track(PATHS["atd_meta"])
P = len(teams)
source_signature = dict(raw_sha256=digest(PATHS["atd_raw"]),
                        meta_sha256=digest(PATHS["atd_meta"]), version=2)
manifest = CACHE / "manifest.json"
cache_ok = (manifest.exists() and json.loads(manifest.read_text()) == source_signature
            and (CACHE / "features.npy").exists() and (CACHE / "padding.npy").exists())
if not cache_ok:
    # Missing positions stay NaN until the explicit feature/mask conversion.
    # Пропуски остаются NaN до явного преобразования признаков и масок.
    xy = np.full((N, P + 1, 2), np.nan, dtype=np.float64)
    with open(PATHS["atd_raw"]) as f:
        count = 0
        for i, line in enumerate(tqdm(f, total=N, desc="Read tracking / Чтение")):
            frame, players, ball = line.strip().split(":")
            assert int(frame) == i + 1
            xy[i, :-1] = np.fromstring(players.replace(";", ","), sep=",").reshape(P, 3)[:, :2]
            xy[i, -1] = np.fromstring(ball, sep=",")[:2]
            count += 1
    assert count == N
    seen = np.isfinite(xy).all(axis=-1)
    keeper = np.zeros((N, P), dtype=bool)
    suspect = np.zeros((N, P), dtype=bool)
    lines = np.full((N, 2), np.nan, dtype=np.float64)
    sides = {}
    for half, first, last in BLOCKS:
        anchor = {1: 11185, 2: 113742}[half]
        centers = [np.nanmedian(np.nanmedian(xy[anchor:anchor+250, :P, 0][:, teams==t], axis=1))
                   for t in (0, 1)]
        assert abs(centers[0]-centers[1]) > .05
        sides[half] = [centers[0] < centers[1], centers[1] < centers[0]]
        rows = np.arange(first, last+1)
        for team in (0, 1):
            slots = np.flatnonzero(teams == team)
            x = xy[first:last+1, :P, 0][:, slots].copy()
            visible = seen[first:last+1, :P][:, slots]
            left = sides[half][team]
            fill = np.inf if left else -np.inf
            x[~visible] = fill
            candidate = x.argmin(axis=1) if left else x.argmax(axis=1)
            rr = np.arange(len(rows))
            depth = (x[rr, candidate] if left else 1-x[rr, candidate]) * LENGTH
            eligible = visible.any(axis=1) & (depth >= -1e-10) & (depth <= 16.5 + 1e-10)
            keeper[rows[eligible], slots[candidate[eligible]]] = True
            x[rr[eligible], candidate[eligible]] = fill
            line = x.min(axis=1) if left else x.max(axis=1)
            line[visible.sum(axis=1)-eligible.astype(int) == 0] = np.nan
            lines[rows, team] = line
        for team in (0, 1):
            slots = np.flatnonzero(teams == team)
            x = xy[first:last+1, :P, 0][:, slots]
            line = lines[first:last+1, 1-team, None]
            behind = x < line if sides[half][1-team] else x > line
            suspect[np.ix_(rows, slots)] = seen[first:last+1, :P][:, slots] & np.isfinite(line) & behind
    features = np.lib.format.open_memmap(CACHE / "features.npy", mode="w+", dtype="float32", shape=(N,P+1,11))
    padding = np.lib.format.open_memmap(CACHE / "padding.npy", mode="w+", dtype="bool", shape=(N,P+1))
    for first in tqdm(range(0, N, 2048), desc="Features / Признаки"):
        end = min(first+2048, N)
        present = seen[first:end]
        bseen = present[:, -1]
        raw = xy[first:end]
        f = np.zeros((end-first,P+1,11), dtype=np.float32)
        f[..., :2] = np.nan_to_num(raw, nan=0.)
        # Scale relative displacement by pitch dimensions, not sample statistics.
        # Масштаб смещения фиксирован, статистики выборки не нужны.
        delta = raw - raw[:, -1:, :]
        f[..., 2:4] = np.where((present & bseen[:,None])[...,None], delta, 0.)
        f[..., 4] = bseen[:,None]
        f[:, :P, 5] = teams == 0
        f[:, :P, 6] = teams == 1
        f[:, -1, 7] = 1
        f[:, :P, 8] = keeper[first:end]
        f[:, :P, 9] = suspect[first:end]
        f[..., 10] = present
        features[first:end] = f
        padding[first:end] = ~present
        padding[first:end, -1] = False  # Keep explicit missing-ball token / Оставляем токен мяча
    features.flush(); padding.flush()
    save_json(manifest, source_signature)  # Written last / Записываем только после завершения
    del xy, seen, keeper, suspect, lines, features, padding
features = np.load(CACHE / "features.npy", mmap_mode="r")
padding = np.load(CACHE / "padding.npy", mmap_mode="r")
assert features.shape == (N, P+1, 11)

# Only the PUBLIC truth file is opened. No private truth or proxy labels.
# Читаем только публичную разметку; закрытая разметка не используется.
gt_path = ROOT / "evaluation/gt/BAR-ATH_possession_gt_public.csv"
gt = pd.read_csv(gt_path)
assert np.array_equal(gt.frame.to_numpy(), np.arange(N))
y = np.full(N, -1, dtype=np.int64)
for k, label in enumerate(LABELS):
    y[gt.label.to_numpy() == label] = k
labeled = np.flatnonzero(y >= 0)
assert len(labeled) == 300 * FPS and np.all(np.diff(labeled) == 1)
START, END = int(labeled[0]), int(labeled[-1]+1)
SPLIT = START + 240 * FPS
TRAIN_END, VAL_END = (START+64, SPLIT+64) if SMOKE else (SPLIT, END)
assert TRAIN_END <= SPLIT < VAL_END <= END
save_json(OUT / "data_signature.json", dict(**source_signature, public_gt_sha256=digest(gt_path)))
print("Train frames:", TRAIN_END-START, "| Validation:", VAL_END-SPLIT)


## Модель и функция потерь
Два слоя Transformer Encoder (64 числа, 4 головы), токен сбора информации, однослойная GRU(64), линейный выход H/A/D.
Нет позиционных номеров игроков: перестановка токенов не меняет предсказание.
Во времени GRU работает только слева направо. Обучение блоками по 125 кадров с переносом скрытого состояния и отсечением градиента на границе блока (TBPTT).

`loss = cross_entropy + 0.05 × mean(sum(abs(p[t] - p[t-1])))` только для соседних кадров с одинаковой истинной меткой. Переходы разметки не штрафуются. У границ блоков прошлый прогноз отсоединён от графа, но участвует в штрафе.


In [ ]:
class PossessionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.input = nn.Linear(11, 64)
        self.cls = nn.Parameter(torch.zeros(1, 1, 64))
        layer = nn.TransformerEncoderLayer(64, 4, dim_feedforward=128, dropout=.1,
                                           activation="gelu", batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, 2, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(64)
        self.gru = nn.GRU(64, 64, batch_first=True)
        self.head = nn.Linear(64, 3)
        # Distinct initial weights across encoder layers / Разная инициализация слоёв
        for module in self.encoder.layers:
            for p in module.parameters():
                if p.ndim > 1:
                    nn.init.xavier_uniform_(p)

    def forward(self, x, mask, hidden=None):
        # Each frame is a set; the first dimension here is time, not player identity.
        # Каждый кадр — множество; первая ось здесь время, а не игроки.
        tokens = self.input(x)
        cls = self.cls.expand(len(x), -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        mask = torch.cat([torch.zeros((len(x),1), dtype=torch.bool, device=x.device), mask], dim=1)
        embedding = self.norm(self.encoder(tokens, src_key_padding_mask=mask)[:, 0])
        temporal, hidden = self.gru(embedding.unsqueeze(0), hidden)
        return self.head(temporal.squeeze(0)), hidden

def tensor_block(first, end):
    # Copy memory-map slices so PyTorch never mutates read-only arrays.
    # Копия среза защищает исходный массив от изменений.
    return (torch.tensor(np.array(features[first:end]), device=DEVICE),
            torch.tensor(np.array(padding[first:end]), device=DEVICE))

def objective(logits, target, previous_p=None, previous_y=None):
    ce = nn.functional.cross_entropy(logits, target)
    probabilities = logits.softmax(-1)
    pp, yy = probabilities, target
    if previous_p is not None:
        pp = torch.cat([previous_p[None], pp])
        yy = torch.cat([previous_y[None], yy])
    same = yy[1:] == yy[:-1]
    variation = (pp[1:] - pp[:-1]).abs().sum(-1)
    smooth = variation[same].mean() if same.any() else probabilities.sum() * 0
    return ce + SMOOTH_WEIGHT*smooth, ce, smooth

@torch.no_grad()
def predict(model, first, end, show=True):
    model.eval()
    hidden = None
    result = np.empty(end-first, dtype=np.int64)
    for a in tqdm(range(first, end, SEQ_LEN), desc="Predict / Прогноз", disable=not show, leave=False):
        b = min(a+SEQ_LEN, end)
        logits, hidden = model(*tensor_block(a,b), hidden)
        result[a-first:b-first] = logits.argmax(-1).cpu().numpy()
        if show:
            save_json(OUT / "progress.json", dict(status="inference", first_frame=first,
                      frames_done=b-first, frames_total=end-first))
    return result

def score(truth, prediction):
    runs = metrics.valid_runs(truth)
    return dict(macroF1=metrics.macro_f1(metrics.scored_confusion(metrics.confusion(truth, prediction))),
                spellF1=metrics.spell_scores(truth, prediction, runs)["mean_f1"],
                switches=int(np.count_nonzero(np.diff(prediction))))

# Structural checks: missing objects, permutation invariance, finite gradients.
# Проверяем пропуски, независимость от порядка игроков и конечность градиентов.
probe = PossessionNet().to(DEVICE).eval()
x, mask = tensor_block(START, START+8)
with torch.no_grad():
    a, _ = probe(x, mask)
    perm = torch.randperm(P+1, device=DEVICE)
    b, _ = probe(x[:,perm], mask[:,perm])
    torch.testing.assert_close(a, b, atol=2e-6, rtol=2e-5)
    empty_mask = torch.ones_like(mask); empty_mask[:,-1] = False
    empty_logits, _ = probe(torch.zeros_like(x), empty_mask)
    assert torch.isfinite(empty_logits).all()
# Temporal chunking preserves inference; later inputs cannot affect earlier answers.
# Разбиение на блоки сохраняет прогноз; будущие кадры не меняют прошлый ответ.
with torch.no_grad():
    prefix, h = probe(x[:4], mask[:4])
    suffix, _ = probe(x[4:], mask[4:], h)
    torch.testing.assert_close(a, torch.cat([prefix,suffix]), atol=2e-6, rtol=2e-5)
    changing = torch.tensor([[5.,0.,0.],[0.,5.,0.]], device=DEVICE)
    _, _, transition_penalty = objective(changing, torch.tensor([0,1], device=DEVICE))
    _, _, stable_penalty = objective(changing, torch.tensor([0,0], device=DEVICE))
    assert transition_penalty.item() == 0 and stable_penalty.item() > 0
probe.train()
logits, _ = probe(x,mask)
loss, _, _ = objective(logits, torch.tensor(y[START:START+8], device=DEVICE))
loss.backward()
assert all(torch.isfinite(p.grad).all() for p in probe.parameters() if p.grad is not None)
assert any(p.grad is not None and p.grad.abs().sum()>0 for p in probe.encoder.parameters())
print("Checks passed | Parameters:", sum(p.numel() for p in probe.parameters()))
del probe


## Обучение с прогрессом и сохранением
Состояние GRU переносится между последовательными блоками, сбрасывается в начале эпохи. Проверка последней минуты начинается с пустого состояния, без доступа к обучающим кадрам.
Количество эпох фиксировано заранее: проверочная минута не выбирает «лучшую» модель. Финальная модель обучается заново, на всех пяти минутах.
Если остановить ноутбук посреди эпохи, повторный запуск повторит только незавершённую эпоху. Не запускайте один RUN_NAME одновременно в двух ядрах.


In [ ]:
def train(stage, first, end, validate=False):
    folder = OUT / stage
    folder.mkdir(exist_ok=True)
    checkpoint = folder / "last.pt"
    seed_all()
    model = PossessionNet().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=.01)
    history, initial_epoch = [], 0
    if RESUME and checkpoint.exists():
        # Only load our own locally created checkpoint / Только собственный локальный checkpoint
        state = torch.load(checkpoint, map_location=DEVICE, weights_only=False)
        assert state["config"] == config and state["data_signature"] == json.loads((OUT / "data_signature.json").read_text())
        assert state["span"] == [first,end]
        model.load_state_dict(state["model"]); optimizer.load_state_dict(state["optimizer"])
        history, initial_epoch = state["history"], state["epoch"]
        torch.set_rng_state(state["torch_rng"].cpu())
        if DEVICE.startswith("cuda"):
            torch.cuda.set_rng_state_all(state["cuda_rng"])
        print(stage, "resuming after epoch", initial_epoch)
    for epoch in tqdm(range(initial_epoch, EPOCHS), desc=stage + " / Эпохи"):
        model.train()
        hidden, previous_p, previous_y = None, None, None
        total_loss, total_ce, total_smooth, frames = 0., 0., 0., 0
        started = time.time()
        blocks = tqdm(range(first, end, SEQ_LEN), desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)
        for a in blocks:
            b = min(a+SEQ_LEN, end)
            optimizer.zero_grad(set_to_none=True)
            target = torch.tensor(y[a:b], device=DEVICE)
            logits, hidden = model(*tensor_block(a,b), hidden)
            loss, ce, smooth = objective(logits, target, previous_p, previous_y)
            assert torch.isfinite(loss), "Non-finite training loss"
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            # Carry memory, not the graph / Переносим память, но не граф градиентов
            hidden = hidden.detach()
            previous_p, previous_y = logits[-1].softmax(-1).detach(), target[-1].detach()
            size = b-a; frames += size
            total_loss += loss.item()*size; total_ce += ce.item()*size; total_smooth += smooth.item()*size
            blocks.set_postfix(loss=f"{total_loss/frames:.4f}")
            save_json(OUT / "progress.json", dict(stage=stage, status="training", epoch=epoch+1,
                      epochs=EPOCHS, frames_done=frames, frames_total=end-first, loss=total_loss/frames))
        row = dict(epoch=epoch+1, loss=total_loss/frames, ce=total_ce/frames,
                   smooth=total_smooth/frames, seconds=time.time()-started)
        if validate:
            prediction = predict(model, SPLIT, VAL_END, show=False)
            row.update({"val_"+k:v for k,v in score(y[SPLIT:VAL_END], prediction).items()})
        history.append(row)
        state = dict(model=model.state_dict(), optimizer=optimizer.state_dict(), epoch=epoch+1,
                     history=history, config=config, span=[first,end], torch_rng=torch.get_rng_state(),
                     cuda_rng=torch.cuda.get_rng_state_all() if DEVICE.startswith("cuda") else [],
                     data_signature=json.loads((OUT / "data_signature.json").read_text()))
        temporary = checkpoint.with_suffix(".tmp")
        torch.save(state, temporary); temporary.replace(checkpoint)
        pd.DataFrame(history).to_csv(folder / "history.csv", index=False)
        tqdm.write(f"{stage}: epoch {epoch+1}/{EPOCHS}, loss={row['loss']:.4f}" +
                   (f", val spellF1={row['val_spellF1']:.4f}" if validate else ""))
    pd.DataFrame(history).to_csv(folder / "history.csv", index=False)
    save_json(OUT / "progress.json", dict(stage=stage, status="completed", epochs=EPOCHS))
    return model, pd.DataFrame(history)

diagnostic_model, diagnostic_history = train("diagnostic_4min", START, TRAIN_END, validate=True)
validation_prediction = predict(diagnostic_model, SPLIT, VAL_END)
validation_scores = score(y[SPLIT:VAL_END], validation_prediction)
save_json(OUT / "validation_metrics.json", validation_scores)
pd.DataFrame({"frame": np.arange(SPLIT,VAL_END), "label": LABELS[validation_prediction]}).to_csv(
    OUT / "validation_predictions.csv", index=False)
print("Diagnostic only / Только диагностика:", validation_scores)


In [ ]:
# Curves are local diagnostics, not publishable training-data exports.
# Графики — локальная диагностика, не публикуем их вместе с кодом.
fig, axes = plt.subplots(1,2,figsize=(11,3))
axes[0].plot(diagnostic_history.epoch, diagnostic_history.loss, label="train loss")
axes[1].plot(diagnostic_history.epoch, diagnostic_history.val_spellF1, label="validation spellF1")
axes[1].plot(diagnostic_history.epoch, diagnostic_history.val_macroF1, label="validation macroF1")
for ax in axes:
    ax.set_xlabel("Epoch / Эпоха"); ax.legend(); ax.grid(alpha=.2)
fig.tight_layout(); fig.savefig(OUT / "learning_curves.png", dpi=140)
plt.show()

if RUN_FINAL:
    final_model, final_history = train("final_5min", START, END)
    print("Final weights / Финальные веса:", OUT / "final_5min/last.pt")
else:
    print("Final refit disabled / Финальное обучение выключено")


## Прогноз всего матча
Модель обрабатывает каждый тайм отдельно, с переносом памяти между блоками. Вне таймов пишем X.
При повторном запуске без обучения можно оставить RESUME=True: завершённые эпохи будут пропущены.
`submission.csv` — финальный файл ответа. `validation_predictions.csv` — только короткий диагностический отрезок, не submission.
Никакого дополнительного сглаживания или ручных исправлений предсказаний.


In [ ]:
if RUN_FULL_MATCH and RUN_FINAL:
    answer = np.full(N, "X", dtype="<U1")
    for half, first, last in BLOCKS:
        save_json(OUT / "progress.json", dict(status="inference", half=half))
        answer[first:last+1] = LABELS[predict(final_model, first, last+1)]
    result = pd.DataFrame({"frame":np.arange(N), "label":answer})
    assert len(result) == N and result.label.isin(["H","A","D","X"]).all()
    error = metrics.validate_submission(result, N)
    assert error is None, error
    result.to_csv(OUT / "submission.csv", index=False)
    save_json(OUT / "progress.json", dict(status="completed", output="submission.csv",
              sha256=digest(OUT / "submission.csv")))
    print("Saved / Сохранено:", OUT / "submission.csv")
else:
    print("Full-match inference disabled / Прогноз полного матча выключен")
